# EduRead — Gemma 3 for Accessible Alt-Text & Mathematical Descriptions
**End-to-end Google Colab capstone notebook**

This notebook provides:
1. Secure setup and hardware checks
2. A dataset schema and synthetic *pipeline-test* images (not research ground truth)
3. Gemma 3 baseline inference
4. Dataset validation and split preparation
5. A guarded fine-tuning section with explicit compatibility checks
6. Baseline evaluation and human-review templates
7. A Gradio accessibility demo
8. Export of artifacts

**Important:** This notebook is a reproducible prototype scaffold. It does not claim that a fine-tuned model was trained. Multimodal Gemma fine-tuning APIs and GPU memory needs depend on the exact software versions and GPU available in your Colab session. The training section deliberately stops unless you provide reviewed image-reference data and verify a compatible multimodal trainer. Do not use synthetic examples as ground truth or deploy outputs for high-stakes learning without review.

## Before you start
1. Open this notebook in Google Colab.
2. Choose **Runtime → Change runtime type → T4 GPU**, if available. Free GPU availability is not guaranteed.
3. On Hugging Face, open the [Gemma 3 4B Instruct model page](https://huggingface.co/google/gemma-3-4b-it), review/accept its terms if prompted, and confirm account access.
4. In Colab's **Secrets** panel, add a secret named `HF_TOKEN` only if required, and grant this notebook access. Never paste the token into code or print it.
5. Run cells top to bottom. If dependency upgrades conflict, restart the runtime and rerun from the start.

Use only images you created, have permission to use, or whose license allows your intended use. Keep private student information out of the notebook and public demo.

In [ ]:
# 1. Install baseline and app dependencies
%pip install -q -U "transformers>=4.50.0" accelerate pillow gradio pandas scikit-learn sympy huggingface_hub peft trl bitsandbytes datasets

In [ ]:
# 2. Hardware and environment check
import os, sys, platform, json, random, re, time
from pathlib import Path
import torch

print("Python:", sys.version.split()[0])
print("OS:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print("No GPU found. A 4B multimodal model may be impractically slow or run out of memory on CPU.")

In [ ]:
# 3. Optional Hugging Face login via Colab Secrets
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Hugging Face authentication completed; token hidden.")
else:
    print("No HF_TOKEN found. Public access may work if model terms are already accepted.")

In [ ]:
# 4. Create project folders
PROJECT = Path("/content/eduread")
IMAGE_DIR = PROJECT / "data" / "images"
OUT_DIR = PROJECT / "outputs"
for folder in [IMAGE_DIR, OUT_DIR, PROJECT / "data"]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT)

## Dataset preparation

The manifest uses one JSONL record per image. A record must point to a real image and contain a human-verified reference before it can be considered training data. Start with diverse categories:
- short alt-text;
- detailed visual/spatial description;
- equation transcription;
- graph/plot description;
- table or scientific diagram description;
- uncertainty handling for unreadable or ambiguous labels.

Split by source/page/diagram family to reduce leakage. Keep a test set untouched until final evaluation. The tiny synthetic images below are only for testing the notebook's image handling and interface; their labels are not a substitute for a reviewed dataset.

In [ ]:
# 5. Create two synthetic pipeline-test images and a starter manifest
from PIL import Image, ImageDraw
import pandas as pd

# A simple triangle drawing for testing file handling only.
im = Image.new("RGB", (640, 420), "white")
d = ImageDraw.Draw(im)
d.line([(130, 330), (130, 90), (470, 330), (130, 330)], fill="black", width=5)
d.text((100, 345), "A", fill="black")
d.text((105, 65), "B", fill="black")
d.text((480, 335), "C", fill="black")
d.text((145, 210), "3", fill="black")
d.text((290, 340), "4", fill="black")
d.text((300, 190), "5", fill="black")
im.save(IMAGE_DIR / "synthetic_triangle.png")

# Simple equation card for pipeline testing only.
eq = Image.new("RGB", (640, 180), "white")
ImageDraw.Draw(eq).text((30, 60), "x + 2 = 5", fill="black")
eq.save(IMAGE_DIR / "synthetic_equation.png")

records = [
    {
        "id": "synthetic_triangle",
        "image": "data/images/synthetic_triangle.png",
        "task": "detailed_description",
        "instruction": "Describe the visible geometry diagram for a visually impaired student.",
        "reference": "",
        "split": "train",
        "review_status": "synthetic_pipeline_test_only",
        "source_license": "created in notebook; pipeline test only"
    },
    {
        "id": "synthetic_equation",
        "image": "data/images/synthetic_equation.png",
        "task": "math_transcription",
        "instruction": "Transcribe the visible equation and explain it accessibly.",
        "reference": "",
        "split": "validation",
        "review_status": "synthetic_pipeline_test_only",
        "source_license": "created in notebook; pipeline test only"
    }
]
MANIFEST = PROJECT / "data" / "manifest.jsonl"
with MANIFEST.open("w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("Created test images and manifest:", MANIFEST)

In [ ]:
# 6. Load and validate the manifest
def load_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]

manifest = load_jsonl(MANIFEST)
validation_rows = []
for r in manifest:
    image_path = PROJECT / r["image"]
    validation_rows.append({
        "id": r.get("id"),
        "task": r.get("task"),
        "image_exists": image_path.is_file(),
        "reference_present": bool(str(r.get("reference", "")).strip()),
        "split": r.get("split"),
        "review_status": r.get("review_status"),
    })
display(pd.DataFrame(validation_rows))
print("Edit /content/eduread/data/manifest.jsonl and add your reviewed examples before training.")

### Add your own data
For each image, add a manifest record like this (the reference must be checked against that exact image):

```json
{"id":"geo_001","image":"data/images/geo_001.png","task":"short_alt_text","instruction":"Write concise alt-text grounded only in visible details.","reference":"Human-verified description here.","split":"train","review_status":"reviewed","source_license":"created by project team"}
```

Upload image files into `/content/eduread/data/images/`, and edit the manifest. Do not use model-generated predictions as reference text without human correction. Aim initially for a reviewed pilot set of 50–100 images; this is a practical starting target, not a guarantee of statistical significance.

In [ ]:
# 7. Upload image files into the notebook runtime (optional)
# Uploaded files go to /content by default; this cell copies image files into the project images folder.
try:
    from google.colab import files
    uploaded = files.upload()
    for filename in uploaded:
        src = Path("/content") / filename
        if src.is_file() and src.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}:
            dest = IMAGE_DIR / src.name
            if src.resolve() != dest.resolve():
                dest.write_bytes(src.read_bytes())
            print("Copied image:", dest)
except Exception as e:
    print("Colab upload widget unavailable in this environment:", type(e).__name__)

## Baseline: load Gemma 3 and describe an image

This uses the Transformers multimodal chat-template interface. Model loading can fail if access has not been granted, package versions are incompatible, or the available GPU memory is insufficient. Fix the baseline before attempting fine-tuning.

In [ ]:
# 8. Load Gemma 3 baseline
import transformers
print("Transformers:", transformers.__version__)

from transformers import AutoProcessor, Gemma3ForConditionalGeneration

MODEL_ID = "google/gemma-3-4b-it"
processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = Gemma3ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
    token=HF_TOKEN,
)
model.eval()
print("Loaded:", MODEL_ID)

In [ ]:
# 9. Reusable inference function
EDUREAD_PROMPT = '''
You are EduRead, an educational accessibility assistant.
Describe the supplied image for a visually impaired student.

Return:
SHORT ALT-TEXT: one concise sentence.
DETAILED DESCRIPTION: important visible objects, labels, layout, and spatial relationships.
MATHEMATICAL DESCRIPTION: if relevant, explain visible equations, symbols, graph axes, plotted values, tables, or geometry.

Rules:
- State only details supported by the image.
- Do not invent labels, values, or relationships.
- Explicitly flag unreadable or ambiguous symbols.
- Separate observations from mathematical inferences.
- Use clear wording suitable for text-to-speech.
'''

def eduread_generate(image, prompt=EDUREAD_PROMPT, max_new_tokens=300):
    if isinstance(image, (str, Path)):
        image = Image.open(image).convert("RGB")
    else:
        image = image.convert("RGB")
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ],
    }]
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = {k: (v.to(model.device) if hasattr(v, "to") else v) for k, v in inputs.items()}
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    input_ids = inputs.get("input_ids")
    if input_ids is not None:
        output = output[:, input_ids.shape[1]:]
    return processor.batch_decode(output, skip_special_tokens=True)[0].strip()

In [ ]:
# 10. Run baseline on a test image
test_image = IMAGE_DIR / "synthetic_triangle.png"
baseline_text = eduread_generate(test_image)
print(baseline_text)

In [ ]:
# 11. Run baseline over available manifest images and save outputs
predictions = []
for r in manifest:
    img_path = PROJECT / r["image"]
    if not img_path.is_file():
        print("Skipping missing image:", r["id"])
        continue
    try:
        prediction = eduread_generate(img_path)
        predictions.append({
            "id": r["id"], "task": r["task"], "split": r["split"],
            "image": r["image"], "prediction": prediction,
            "reference": r.get("reference", ""),
            "review_status": r.get("review_status", "")
        })
        print("Completed:", r["id"])
    except Exception as e:
        print("Inference failed for", r["id"], type(e).__name__, str(e)[:250])

PRED_PATH = OUT_DIR / "baseline_predictions.jsonl"
with PRED_PATH.open("w", encoding="utf-8") as f:
    for p in predictions:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")
print("Saved:", PRED_PATH)

## Fine-tuning readiness and training

The next cells include an **optional QLoRA/SFT training attempt** using a custom multimodal collator, based on the Hugging Face TRL vision-language fine-tuning workflow. It only runs when the manifest contains enough human-reviewed, real image-reference pairs and a CUDA GPU is available.

Free Colab GPU memory and library compatibility vary. If the trainer fails to initialize or runs out of memory, record the exact error; do not claim fine-tuning succeeded. The custom collator is a starting implementation that must be tested with your installed package versions and a tiny dataset before longer training.

Official reference: https://huggingface.co/docs/trl/main/training_vlm_sft

In [ ]:
# 12. Strict readiness check: only reviewed real image-reference pairs qualify
manifest = load_jsonl(MANIFEST)
def record_is_reviewed_train(r):
    return (
        r.get("split") == "train"
        and r.get("review_status") == "reviewed"
        and bool(str(r.get("reference", "")).strip())
        and (PROJECT / r["image"]).is_file()
        and r.get("review_status") != "synthetic_pipeline_test_only"
    )

def record_is_reviewed_validation(r):
    return (
        r.get("split") in {"validation", "test"}
        and r.get("review_status") == "reviewed"
        and bool(str(r.get("reference", "")).strip())
        and (PROJECT / r["image"]).is_file()
    )

train_records = [r for r in manifest if record_is_reviewed_train(r)]
eval_records = [r for r in manifest if record_is_reviewed_validation(r)]
print("Eligible reviewed train examples:", len(train_records))
print("Eligible reviewed validation/test examples:", len(eval_records))
if len(train_records) < 10 or len(eval_records) < 3:
    print("NOT READY: add more reviewed image-reference pairs before fine-tuning.")
else:
    print("Data-count gate passed. Still verify a compatible Gemma 3 multimodal training recipe and GPU memory.")

### Recommended LoRA experiment protocol
1. Start with 2–4 reviewed image-reference examples to check that loss decreases and the model can overfit them.
2. Use the official Gemma 3 multimodal training recipe compatible with the exact installed package versions; do not substitute a text-only trainer.
3. Save only adapter weights and training configuration where licensing permits.
4. Compare baseline and adapted model on the same held-out test set.
5. Stop if loss becomes NaN/Inf, outputs collapse, or GPU memory is exceeded; record the error and environment.
6. Never claim fine-tuning succeeded unless a training run completed and the resulting adapter was saved and tested.

### Optional training cell — QLoRA for reviewed image/description pairs
This cell deliberately refuses to train on synthetic examples, missing images, or unreviewed targets. For a meaningful experiment, use a larger reviewed dataset and keep test examples out of training.


## Evaluation

Use exact match / character similarity only as lightweight diagnostics. Human review is essential because a fluent description can still misstate a label or mathematical relationship. For math transcription, independently verify equations and numerical calculations.

In [ ]:
# 13. Lightweight metrics for predictions with non-empty human references
from difflib import SequenceMatcher

def norm(s):
    return re.sub(r"\s+", " ", str(s or "").lower().strip())

metric_rows = []
for p in predictions:
    ref, pred = p.get("reference", ""), p.get("prediction", "")
    if str(ref).strip() and str(pred).strip():
        metric_rows.append({
            "id": p["id"], "task": p["task"],
            "exact_match": float(norm(pred) == norm(ref)),
            "character_similarity": SequenceMatcher(None, norm(pred), norm(ref)).ratio()
        })
metrics_df = pd.DataFrame(metric_rows)
if not metrics_df.empty:
    display(metrics_df)
    display(metrics_df.groupby("task")[["exact_match", "character_similarity"]].mean())
else:
    print("No reviewed references available for automatic metrics. Add references before interpreting evaluation.")

In [ ]:
# 14. Export a human-evaluation template
review_columns = [
    "id", "task", "prediction", "grounding_1to5", "math_correctness_1to5",
    "coverage_1to5", "conciseness_1to5", "accessibility_1to5",
    "uncertainty_handling_1to5", "reviewer_notes"
]
review_rows = []
for p in predictions:
    row = {"id": p.get("id", ""), "task": p.get("task", ""), "prediction": p.get("prediction", "")}
    row.update({k: "" for k in review_columns[3:]})
    review_rows.append(row)
review_csv = OUT_DIR / "human_evaluation_template.csv"
pd.DataFrame(review_rows, columns=review_columns).to_csv(review_csv, index=False)
print("Human review CSV:", review_csv)

### Human evaluation rubric (score 1–5)
- **Grounding:** no invented objects, labels, values, or relationships.
- **Mathematical correctness:** transcription and mathematical claims are correct.
- **Coverage:** includes information needed to understand the visual.
- **Conciseness:** short alt-text avoids unnecessary detail.
- **Accessibility:** understandable when heard through a screen reader.
- **Uncertainty handling:** flags unreadable or ambiguous details.

Report sample size, category distribution, errors, and limitations. Keep the test set untouched until final evaluation.

## Accessible Gradio application

The UI provides an image input, a choice of description style, and a text output. Test keyboard navigation and a real screen reader (for example, NVDA on Windows) before claiming accessibility. This is a prototype, not a certified accessibility product.

In [ ]:
# 15. Define Gradio app
import gradio as gr

PROMPTS = {
    "Concise alt-text": "Write one concise sentence of alt-text grounded only in visible details. Flag uncertainty.",
    "Detailed description": "Describe visible objects, labels, layout, and spatial relationships for a visually impaired student. Do not invent details; flag uncertainty.",
    "Mathematical description": "Transcribe and explain visible mathematical symbols, equations, graphs, or geometry. Preserve notation, separate observation from inference, and flag unreadable details.",
    "Full accessible description": EDUREAD_PROMPT,
}

def describe_ui(image, style):
    if image is None:
        return "Please upload an educational image."
    try:
        return eduread_generate(image, prompt=PROMPTS[style], max_new_tokens=300)
    except Exception as e:
        return f"Could not generate a description ({type(e).__name__}). Check model loading, memory, and runtime logs."

with gr.Blocks(title="EduRead — Accessible Educational Image Descriptions") as demo:
    gr.Markdown("# EduRead\nUpload an educational image to generate an accessible description.")
    image_input = gr.Image(type="pil", label="Educational image")
    style = gr.Radio(
        choices=list(PROMPTS.keys()),
        value="Full accessible description",
        label="Description type"
    )
    run_button = gr.Button("Generate description")
    output_box = gr.Textbox(label="Generated description", lines=12, interactive=False)
    gr.Markdown("**Please verify:** AI may misread mathematical symbols and diagram labels. Confirm the result against the original image.")
    run_button.click(describe_ui, inputs=[image_input, style], outputs=output_box)
print("Gradio app defined.")

In [ ]:
# 16. Launch app
# share=True creates a temporary public URL while the Colab session is running.
# Do not use confidential student images. Set share=False for a private local notebook session.
demo.launch(share=False, debug=False)

## Save and download artifacts

Colab storage is temporary. Download outputs or copy the project directory to Google Drive before ending the session. Do not include tokens, private datasets, or model weights in public repositories.

In [ ]:
# 17. Download outputs and manifest as a ZIP
import shutil
zip_path = shutil.make_archive("/content/EduRead_artifacts", "zip", root_dir=PROJECT)
print("Created:", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Download manually from:", zip_path)

## Final capstone checklist
- [ ] Model access and terms confirmed.
- [ ] Environment and GPU recorded.
- [ ] Dataset source/license documented.
- [ ] Human-verified references used for training/evaluation.
- [ ] Train/validation/test split avoids near-duplicate leakage.
- [ ] Baseline predictions saved.
- [ ] Multimodal fine-tuning recipe verified for the exact package versions.
- [ ] Adapter successfully saved and tested before claiming fine-tuning.
- [ ] Baseline vs adapted model compared on identical held-out images.
- [ ] Mathematical errors and hallucinations reviewed.
- [ ] Keyboard and screen-reader interaction tested.
- [ ] Secrets, private images, and model weights excluded from public GitHub.

**Suggested project description:** “EduRead is a prototype exploring Gemma 3 for educational image accessibility. Its effectiveness is assessed using human-reviewed descriptions and mathematical correctness checks.” Claim improvements only when supported by your measured results.